In [ ]:
# ========================================
# ROBUST TRAINING - ULTRA-AGGRESSIVE AUGMENTATION
# ========================================
# Target: Robust AUC 0.93-0.96 for CodaBench competition
# Strategy: Extreme degradation during training to maximize robustness

import sys
sys.path.append('../')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from src.dataset import NTIRETrainDataset, TransformSubset, get_transforms
from src.models import CLIPDetector
import time
from datetime import datetime
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

print("=" * 60)
print("ROBUST TRAINING - ULTRA-AGGRESSIVE AUGMENTATION")
print("=" * 60)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)
print("\n🎯 Target: Maximize Robust ROC AUC")
print("   Current: Clean 0.96, Robust 0.85 (gap 11%)")
print("   Goal:    Clean 0.93, Robust 0.94 (gap 1%)")
print("=" * 60)

# ============================================
# CONFIGURATION - OPTIMIZED FOR ROBUSTNESS
# ============================================

# Training hyperparameters
BATCH_SIZE = 12              # Smaller batch for degraded images
EPOCHS = 8                   # More epochs for convergence
LR = 5e-5                    # Lower learning rate
ACCUMULATION_STEPS = 3       # Effective batch = 36
VAL_SPLIT = 0.1
WEIGHT_DECAY = 1e-3          # More regularization

# Device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"\n💻 Device: {device}")

# ============================================
# LOAD DATASET WITH ULTRA-AGGRESSIVE AUGMENTATION
# ============================================

print("\n📂 Loading dataset with ULTRA-AGGRESSIVE augmentation...")
train_transform, val_transform = get_transforms()

base_dataset = NTIRETrainDataset(
    '../data/train',
    shard_nums=None,  # All 6 shards (277k images)
    transform=None
)

print(f"\n📊 Dataset loaded:")
print(f"  Total images: {len(base_dataset):,}")
print(f"  Real: {(base_dataset.label_df['label'] == 0).sum():,}")
print(f"  Fake: {(base_dataset.label_df['label'] == 1).sum():,}")

# Split indices
val_size = int(len(base_dataset) * VAL_SPLIT)
train_size = len(base_dataset) - val_size

indices = list(range(len(base_dataset)))
train_indices, val_indices = random_split(
    indices,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Create datasets with different transforms
print("\n🔧 Applying transformations...")
print("  Train: ULTRA-AGGRESSIVE augmentation (98% degradation)")
print("  Val:   MODERATE augmentation (70% degradation)")

train_dataset = TransformSubset(base_dataset, train_indices, train_transform)
val_dataset = TransformSubset(base_dataset, val_indices, val_transform)

print(f"\n✓ Dataset split:")
print(f"  Train: {len(train_dataset):,} images")
print(f"  Val:   {len(val_dataset):,} images")

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=False
)

# Training estimates
iterations_per_epoch = len(train_loader)
time_per_iteration = 1.8  # seconds (slower due to augmentation)
time_per_epoch_min = (iterations_per_epoch * time_per_iteration) / 60
total_time_hours = (time_per_epoch_min * EPOCHS) / 60

print(f"\n⏱️  Training Estimates:")
print(f"  Iterations/epoch: {iterations_per_epoch:,}")
print(f"  Time/epoch: ~{time_per_epoch_min:.0f} minutes")
print(f"  Total time: ~{total_time_hours:.1f} hours")

estimated_finish = datetime.now().timestamp() + (total_time_hours * 3600)
finish_time = datetime.fromtimestamp(estimated_finish)
print(f"  Estimated finish: {finish_time.strftime('%Y-%m-%d %H:%M:%S')}")

# ============================================
# MODEL SETUP
# ============================================

print("\n🤖 Creating model...")
model = CLIPDetector()

# Count parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Total parameters: {total_params:,}")

# ============================================
# IMPORTANT WARNINGS
# ============================================

print("\n" + "=" * 60)
print("⚠️  IMPORTANT - READ BEFORE STARTING")
print("=" * 60)
print(f"1. Training will take ~{total_time_hours:.0f} hours (~{total_time_hours/24:.1f} days)")
print("2. Mac will be busy - cannot use for other tasks")
print("3. DISABLE SLEEP MODE:")
print("   System Settings → Battery → Prevent auto sleep")
print("4. Keep Mac plugged in and charging")
print("5. Checkpoints saved every epoch (8 files total)")
print("6. Training optimizes for ROBUST AUC (not Clean AUC)")
print("7. Expect Clean AUC to drop slightly (~0.93)")
print("8. Expect Robust AUC to improve significantly (~0.94)")
print("=" * 60)

# Confirmation
print("\n" + "=" * 60)
response = input("Ready to start aggressive training? Type 'YES' to continue: ")
if response.upper() != 'YES':
    print("Training cancelled.")
    import sys
    sys.exit(0)

print("=" * 60)
print("🚀 TRAINING STARTED")
print("=" * 60)

# ============================================
# TRAINING FUNCTION
# ============================================

def train_robust_model(model, train_loader, val_loader, epochs, lr, device, accumulation_steps):
    """
    Train model with focus on robustness
    Saves checkpoint every epoch
    """
    model = model.to(device)
    
    # Optimizer with higher weight decay
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.98),
        eps=1e-6
    )
    
    # Cosine annealing scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs,
        eta_min=1e-7
    )
    
    criterion = nn.BCEWithLogitsLoss()
    
    best_robust_auc = 0.0
    history = {
        'train_loss': [],
        'val_auc': [],
        'epochs': []
    }
    
    # Open log file
    log_file = open('../checkpoints/robust_training.log', 'w')
    
    def log(msg):
        """Print and save to file"""
        print(msg)
        log_file.write(msg + '\n')
        log_file.flush()
    
    log(f"Training started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    log("=" * 60)
    log(f"Configuration:")
    log(f"  Epochs: {epochs}")
    log(f"  Batch size: {BATCH_SIZE} (effective: {BATCH_SIZE * accumulation_steps})")
    log(f"  Learning rate: {lr}")
    log(f"  Weight decay: {WEIGHT_DECAY}")
    log(f"  Augmentation: ULTRA-AGGRESSIVE (98% train, 70% val)")
    log("=" * 60)
    
    for epoch in range(epochs):
        epoch_start = time.time()
        
        log(f"\n{'='*60}")
        log(f"Epoch {epoch+1}/{epochs}")
        log(f"{'='*60}")
        
        # ============================================
        # TRAINING PHASE
        # ============================================
        model.train()
        train_loss = 0.0
        optimizer.zero_grad()
        
        pbar = tqdm(train_loader, desc=f"Training {epoch+1}/{epochs}")
        for i, (images, labels) in enumerate(pbar):
            images = images.to(device)
            labels = labels.float().to(device)
            
            # Forward pass
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels) / accumulation_steps
            
            # Backward pass
            loss.backward()
            
            # Update weights every accumulation_steps
            if (i + 1) % accumulation_steps == 0:
                optimizer.step()
                optimizer.zero_grad()
            
            train_loss += loss.item() * accumulation_steps
            pbar.set_postfix({
                'loss': f'{loss.item() * accumulation_steps:.4f}',
                'lr': f'{optimizer.param_groups[0]["lr"]:.2e}'
            })
        
        avg_train_loss = train_loss / len(train_loader)
        
        # ============================================
        # VALIDATION PHASE
        # ============================================
        model.eval()
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation"):
                images = images.to(device)
                outputs = model(images).squeeze()
                probs = torch.sigmoid(outputs)
                
                all_preds.extend(probs.cpu().numpy())
                all_labels.extend(labels.numpy())
        
        val_auc = roc_auc_score(all_labels, all_preds)
        
        # Update history
        history['train_loss'].append(avg_train_loss)
        history['val_auc'].append(val_auc)
        history['epochs'].append(epoch + 1)
        
        # Calculate times
        epoch_time = time.time() - epoch_start
        remaining_epochs = epochs - (epoch + 1)
        remaining_time = remaining_epochs * epoch_time / 3600
        
        # Log results
        log(f"\nEpoch {epoch+1}/{epochs} Results:")
        log(f"  Train Loss: {avg_train_loss:.4f}")
        log(f"  Val AUC:    {val_auc:.4f}")
        log(f"  Learning Rate: {optimizer.param_groups[0]['lr']:.2e}")
        log(f"  Epoch time: {epoch_time/60:.1f} minutes")
        log(f"  Remaining:  ~{remaining_time:.1f} hours")
        
        # Save checkpoint EVERY epoch
        checkpoint_path = f'../checkpoints/robust_epoch{epoch+1}.pth'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'auc': val_auc,
            'loss': avg_train_loss,
            'history': history
        }, checkpoint_path)
        log(f"  ✓ Checkpoint saved: {checkpoint_path}")
        
        # Track best ROBUST model
        # Note: This is validation on degraded images!
        if val_auc > best_robust_auc:
            best_robust_auc = val_auc
            best_path = '../checkpoints/robust_best.pth'
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'auc': val_auc,
                'loss': avg_train_loss
            }, best_path)
            log(f"  🏆 New best Robust AUC! Saved to: {best_path}")
        
        # Update learning rate
        scheduler.step()
        
        # Progress summary
        improvement = val_auc - history['val_auc'][0] if epoch > 0 else 0
        log(f"\n  Progress: AUC {history['val_auc'][0]:.4f} → {val_auc:.4f} ({improvement:+.4f})")
    
    log("\n" + "=" * 60)
    log(f"Training completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    log(f"Best Robust AUC: {best_robust_auc:.4f}")
    log("=" * 60)
    
    log_file.close()
    
    return model, history

# ============================================
# START TRAINING
# ============================================

model, history = train_robust_model(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    device=device,
    accumulation_steps=ACCUMULATION_STEPS
)

# ============================================
# TRAINING SUMMARY
# ============================================

print("\n" + "=" * 60)
print("✅ ROBUST TRAINING COMPLETED")
print("=" * 60)
print(f"Total epochs: {EPOCHS}")
print(f"Best val AUC: {max(history['val_auc']):.4f}")
print(f"Best epoch: {history['val_auc'].index(max(history['val_auc'])) + 1}")

print(f"\n📈 AUC Progression:")
for i, (epoch, auc) in enumerate(zip(history['epochs'], history['val_auc'])):
    marker = "🏆" if auc == max(history['val_auc']) else "  "
    improvement = f"({auc - history['val_auc'][0]:+.4f})" if i > 0 else ""
    print(f"  {marker} Epoch {epoch}: {auc:.4f} {improvement}")

print(f"\n📁 Checkpoints saved:")
for i in range(1, EPOCHS+1):
    print(f"  - checkpoints/robust_epoch{i}.pth")
print(f"  - checkpoints/robust_best.pth (best robust AUC)")

print(f"\n📄 Log file: checkpoints/robust_training.log")

print("\n" + "=" * 60)
print("NEXT STEPS")
print("=" * 60)
print("1. Evaluate best model on CodaBench validation")
print("2. Generate submission CSV")
print("3. Upload to CodaBench")
print("4. Expected: Robust AUC 0.93-0.96 🎯")
print("=" * 60)

# Plot training curves
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss curve
ax1.plot(history['epochs'], history['train_loss'], 'b-', linewidth=2, label='Train Loss')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training Loss Over Time', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# AUC curve
ax2.plot(history['epochs'], history['val_auc'], 'g-', linewidth=2, label='Validation AUC')
ax2.axhline(y=max(history['val_auc']), color='r', linestyle='--', alpha=0.5, label=f'Best: {max(history["val_auc"]):.4f}')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('ROC AUC', fontsize=12)
ax2.set_title('Validation AUC Over Time', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()
ax2.set_ylim([0.85, 1.0])

plt.tight_layout()
plt.savefig('../checkpoints/robust_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Training curves saved: checkpoints/robust_training_curves.png")